In [1]:
import os
import duckdb
import pandas as pd
from pathlib import Path
from datetime import datetime, timezone
import pytz
from google.transit import gtfs_realtime_pb2

In [ ]:
os.chdir('..')
# Persistent database — all tables survive between sessions
con = duckdb.connect('data/transit_kpi.duckdb')

# Eastern timezone — handles EDT/EST automatically
EASTERN = pytz.timezone('America/New_York')

# Verify connection
con.sql("SHOW TABLES").df()

,name
0,calendar
1,calendar_dates
2,gtfsrt
3,stop_times
4,trip_delays
5,trips


In [4]:
def eastern_midnight_unix(date_str: str) -> int:
    """
    Convert a service date string (e.g. '2026-04-07') to a Unix timestamp
    representing midnight Eastern time on that date.
    
    Handles DST automatically -- April returns UTC-4, January returns UTC-5.
    """
    d = datetime.strptime(date_str, '%Y-%m-%d')
    midnight_eastern = EASTERN.localize(datetime(d.year, d.month, d.day, 0, 0, 0))
    return int(midnight_eastern.timestamp())

# Verify it works -- April 7th Eastern midnight should be 1775534400
print(f"April 7 Eastern midnight: {eastern_midnight_unix('2026-04-07')}")
print(f"Expected:                 1775534400")

April 7 Eastern midnight: 1775534400
Expected:                 1775534400


In [5]:
archive_dir = Path('data/raw/gtfs_rt')

# Find all .pb files sorted chronologically
pb_files = sorted(archive_dir.glob('*.pb'))

print(f"Files found: {len(pb_files)}")
print(f"First: {pb_files[0].name}")
print(f"Last:  {pb_files[-1].name}")

Files found: 288
First: 20260407T000452Z.pb
Last:  20260408T204522Z.pb


In [6]:
records = []

for pb_file in pb_files:

    # Extract date and timestamp from filename e.g. 20260407T143022Z.pb
    snapshot_ts = datetime.strptime(
        pb_file.stem, "%Y%m%dT%H%M%SZ"
    ).replace(tzinfo=timezone.utc)

    service_date = snapshot_ts.date().isoformat()

    # Compute Eastern midnight unix for this service date
    midnight_unix = eastern_midnight_unix(service_date)

    feed = gtfs_realtime_pb2.FeedMessage()
    with open(pb_file, 'rb') as f:
        feed.ParseFromString(f.read())

    for entity in feed.entity:

        if not entity.HasField('trip_update'):
            continue

        tu = entity.trip_update
        trip_id = tu.trip.trip_id
        route_id = tu.trip.route_id
        schedule_relationship = tu.trip.schedule_relationship

        for stu in tu.stop_time_update:

            # Prefer arrival time, fall back to departure
            if stu.HasField('arrival'):
                predicted_time = stu.arrival.time
            elif stu.HasField('departure'):
                predicted_time = stu.departure.time
            else:
                continue

            records.append({
                'snapshot_ts':           snapshot_ts.isoformat(),
                'service_date':          service_date,
                'midnight_unix':         midnight_unix,
                'trip_id':               trip_id,
                'route_id':              route_id,
                'schedule_relationship': schedule_relationship,
                'stop_sequence':         stu.stop_sequence,
                'stop_id':               stu.stop_id,
                'predicted_unix':        predicted_time,
                'predicted_ts':          datetime.fromtimestamp(
                                             predicted_time,
                                             tz=timezone.utc
                                         ).isoformat()
            })

print(f"Total records: {len(records)}")

Total records: 4390409


In [7]:
df = pd.DataFrame(records)

con.execute("DROP TABLE IF EXISTS gtfsrt")
con.execute("CREATE TABLE gtfsrt AS SELECT * FROM df")

# Verify
print(con.sql("SELECT COUNT(*) as rows FROM gtfsrt").df().to_string())
print(con.sql("SELECT snapshot_ts, service_date, midnight_unix, trip_id, route_id FROM gtfsrt LIMIT 3").df().to_string())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

      rows
0  4390409
                 snapshot_ts service_date  midnight_unix trip_id route_id
0  2026-04-07T00:04:52+00:00   2026-04-07     1775534400  701387      103
1  2026-04-07T00:04:52+00:00   2026-04-07     1775534400  701387      103
2  2026-04-07T00:04:52+00:00   2026-04-07     1775534400  701387      103
